# k-hop reachability — Lane B

**What this notebook does.** It builds and runs the reachability experiment: can a small model, thinking in repeated loops, correctly say whether you can walk from one dot to another through a network — including on cases far harder than it was trained on — and *why* it succeeds or fails.

**How it's organised.** The real code lives as plain `.py` files in the `tinylab` repo. This notebook is the cockpit: it pulls that code, runs it, and holds notes. Each stage opens with a plain-language note, then a thin code cell that calls into the repo. If the runtime dies, nothing important is lost — re-run from the top.

> **Before running:** fill in `REPO_URL` and check the paths in *Setup*. The three cells tagged **TODO(wire-in)** connect to your existing `lablog.py`, `analyze.py`, and recipe-card format — they're left as stubs until those are confirmed.

## Setup

Pull the repo, install the Colab requirements, and record the exact code version for the run manifest.

In [ ]:
# --- pull the tinylab repo -------------------------------------------------
# Option A (preferred): clone from GitHub once the repo is pushed.
REPO_URL   = 'https://github.com/joyjeet-singh/tinylab.git'  # <-- confirm
BRANCH     = 'main'
REPO_DIR   = '/content/tinylab'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Option B (fallback, if not yet on GitHub): keep the repo on Google Drive and
# point REPO_DIR at it after mounting Drive in the next cell. Then skip the
# clone above.

# --- install the Colab (GPU) requirements ----------------------------------
req = os.path.join(REPO_DIR, 'requirements-colab.txt')
if os.path.exists(req):
    subprocess.run(['pip', 'install', '-q', '-r', req], check=True)
else:
    print('No requirements-colab.txt yet — add one (GPU torch build) alongside your Mac pin.')

In [ ]:
# --- mount Drive for run outputs that must survive disconnects -------------
# Your manifest + metrics logs are tiny text files and can instead be committed
# straight into the repo. Use Drive only for anything heavy (e.g. checkpoints).
from google.colab import drive
drive.mount('/content/drive')
RUNS_DIR = '/content/drive/MyDrive/tinylab_runs/reachability'
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# --- live-reload repo edits, put repo on the path, capture the commit ------
%load_ext autoreload
%autoreload 2
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# The exact code version, recorded into every run's manifest so results stay
# traceable even though we're working in a notebook.
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip()
print('code version:', GIT_COMMIT[:10])

## The question, in one sentence

Does baking structure into a small model's reasoning step let it correctly trace paths on harder cases than it saw in training — and can we pin down why, and exactly where it still breaks?

## Stage 1 — the data

Networks of dots joined by lines. Each question: can you walk from dot A to dot B? Difficulty is set by how many steps the shortest walk takes. We train on pairs 1–5 steps apart and test on pairs 6–12 apart — the gap is the test. Dot names are shuffled every example, and we mix in genuinely unreachable pairs so the model can't win by always saying yes.

In [ ]:
from reachability import graphs

DATA_SEED = 0
train = graphs.generate_dataset(2000, graphs.TRAIN_DISTANCES, n_nodes=32, data_seed=DATA_SEED)
test  = graphs.generate_dataset(2000, graphs.TEST_DISTANCES,  n_nodes=32, data_seed=DATA_SEED + 1)

# quick sanity: distance spread + yes/no balance (never trust a batch you haven't looked at)
from collections import Counter
def summarise(name, data):
    yes = [d.distance for d in data if d.reachable]
    print(f'{name}: n={len(data)}  yes={len(yes)}  no={len(data)-len(yes)}  by-distance={dict(sorted(Counter(yes).items()))}')
summarise('train', train)
summarise('test ', test)

## Stage 2 — the model

A small core that reads the network once (the **encoder**), then repeats a reasoning step some number of times (the **loops**). Two things we vary here: the encoder (a flat reader vs. a neighbour-aware one) and the reasoning step (your structured, physics-shaped version vs. a plain learned one). The number of loops is a dial we can change at test time.

*(Fill in once `reachability/model.py` is written — next step.)*

In [ ]:
# TODO(model): from reachability.model import ReachabilityModel
# model = ReachabilityModel(encoder='gnn', reasoning='structured', width=128, layers=2)
pass

## Stage 3 — train one configuration

Pick a recipe card, train, and log every number to disk (never typed by hand). Write your prediction in the note cell **before** you run — the calibration habit.

In [ ]:
# TODO(wire-in): call your existing lablog.py so runs land in your standard
# per-run folder (manifest.json + metrics.jsonl). Record GIT_COMMIT and the
# data hashes into the manifest exactly as your other lanes do.
#
# run = lablog.start_run(RUNS_DIR, recipe='recipes/reach_v1.yaml', git_commit=GIT_COMMIT)
# train_model(model, train, test, run=run)
pass

## Stage 4 — the loops-by-distance sweep

The core diagnostic: a grid with number of loops across the top and true distance down the side, coloured by whether the model got it right. A clean triangle (solved whenever loops ≥ distance) is the headline; dents in it tell you whether a failure is a 'not enough loops' problem or a 'can't represent it at all' problem.

*(Fill in once `reachability/eval_grid.py` is written.)*

In [ ]:
# TODO(sweep): from reachability.eval_grid import loops_by_distance
# grid = loops_by_distance(model, test, loops=[1,2,3,4,5,6,8,10,12,16])
# grid is accuracy per (loops, distance); save to the run folder and plot.
pass

## Stage 5 — read the results

Every headline number is computed from the logs by the scorekeeper, not typed in. Break results down three ways — reachable vs. unreachable, distance band (including 1–2), and the frontier tracker vs. its base rate — because the overall number hides exactly the failures we care about.

In [ ]:
# TODO(wire-in): run your analyze.py over the run folders in RUNS_DIR.
# e.g.  !python {REPO_DIR}/analyze.py --runs {RUNS_DIR}
pass